In [21]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np

# Chargement
transactions = pd.read_csv("../data/transactions.csv")
customers = pd.read_csv("../data/customers.csv")

print("=== TRANSACTIONS : shape ===")
print(transactions.shape)
print(transactions.dtypes)
print(transactions.head())

=== TRANSACTIONS : shape ===
(1837137, 8)
invoice_id          str
customer_id     float64
product_code        str
product_name        str
quantity        float64
unit_price      float64
invoice_date        str
country             str
dtype: object
  invoice_id  customer_id product_code                      product_name  \
0     513574          NaN        22382        LUNCH BAG SPACEBOY DESIGN    
1     609421          NaN        21028          PURPLE GEMSTONE BRACELET   
2     515609          NaN       47591B          SCOTTIES CHILDRENS APRON   
3     501320      15123.0        22334  DINOSAUR PARTY BAG + STICKER SET   
4     521822      12714.0        21933   PINK VINTAGE PAISLEY PICNIC BAG   

   quantity  unit_price         invoice_date         country  
0       2.0        4.21  2010-06-25 15:13:00  United Kingdom  
1       2.0        8.95  2011-10-16 12:20:00  United Kingdom  
2       1.0        1.66  2010-07-13 15:44:00  United Kingdom  
3       8.0        1.65  2010-03-16 09:49:0

Étape 2 : Détection et traitement des anomalies

Transactions.csv

1. customer_id manquants

On commence par identifier les lignes sans customer_id. Ces lignes correspondent souvent à des ventes en magasin (clients anonymes / walk-in) où le client n'a pas été enregistré dans le CRM.

Décision : On les conserve pour l'analyse du chiffre d'affaires global et des produits, mais on les exclut de toute analyse par client (segmentation, RFM, CLV). On crée un flag is_anonymous pour les identifier facilement.

In [23]:
# --- 1. customer_id manquants ---
missing_cid = transactions["customer_id"].isna().sum()
pct_missing = missing_cid / len(transactions) * 100
print(f"customer_id manquants : {missing_cid} ({pct_missing:.1f}%)")

# Regarder à quoi ressemblent ces lignes
print("\n--- Échantillon lignes sans customer_id ---")
print(transactions[transactions["customer_id"].isna()].head(10))

customer_id manquants : 418258 (22.8%)

--- Échantillon lignes sans customer_id ---
   invoice_id  customer_id product_code                      product_name  \
0      513574          NaN        22382        LUNCH BAG SPACEBOY DESIGN    
1      609421          NaN        21028          PURPLE GEMSTONE BRACELET   
2      515609          NaN       47591B          SCOTTIES CHILDRENS APRON   
6      503061          NaN        22381         TOY TIDY PINK RETROSPOT     
9      756479          NaN        35972         PINK METAL CHICKEN HEART    
17     500827          NaN        22228   BUNNY WOODEN PAINTED WITH BIRD    
26     768176          NaN        22990               WHITE CHERRY LIGHTS   
31     513574          NaN        22350             ILLUSTRATED CAT BOWL    
35     762952          NaN        21336  NEW BAROQUE IVORY CUSHION COVER    
38     577358          NaN        37500       TEA TIME TEAPOT IN GIFT BOX   

    quantity  unit_price         invoice_date         country  
0   

In [24]:
transactions["is_anonymous"] = transactions["customer_id"].isna()

 2. Quantités négatives

Les quantités négatives correspondent à des retours ou annulations. On peut le vérifier en observant que les invoice_id associés commencent souvent par la lettre "C" (convention CRM pour les avoirs/annulations).

Impact : Si on ne les traite pas, elles faussent les agrégations (CA, panier moyen, fréquence).

Décision : On les flag via une colonne is_return et on les exclut des analyses de vente. On les conserve néanmoins dans le dataset pour pouvoir calculer les retours nets si besoin.

In [25]:
# --- 2. Quantity négatives ---
neg_qty = transactions[transactions["quantity"] < 0]
print(f"Lignes quantity < 0 : {len(neg_qty)} ({len(neg_qty)/len(transactions)*100:.1f}%)")
print("\n--- Échantillon quantity négatives ---")
print(neg_qty[["invoice_id", "customer_id", "product_code", "quantity", "unit_price"]].head(10))

# Vérifier si les invoice_id commencent par "C" (convention = annulation/retour)
print("\n--- invoice_id commençant par C (retours/annulations) ---")
print(neg_qty["invoice_id"].astype(str).str.startswith("C").value_counts())

Lignes quantity < 0 : 23314 (1.3%)



--- Échantillon quantity négatives ---
    invoice_id  customer_id product_code  quantity  unit_price
20     C548723      15874.0        72741      -3.0        1.45
69     C535276      15021.0        82483      -6.0        5.95
113     548380          NaN        22127     -60.0        0.00
123    C562221      16525.0        22151    -432.0        0.36
224    C573228      16764.0        23148      -1.0        0.83
227    C573219      14415.0        21672      -1.0        1.45
261     508700          NaN        22192      -1.0        0.00
263    C538082      13777.0        21864     -20.0        2.10
387    C536234      17422.0        22760      -1.0       12.75
390    C566468      13319.0        22842      -1.0        5.95

--- invoice_id commençant par C (retours/annulations) ---
invoice_id
True     19493
False     3821
Name: count, dtype: int64


In [26]:
transactions["is_return"] = transactions["quantity"] < 0

 3. unit_price à zéro

On identifie les lignes où le prix unitaire est nul.

Ces cas peuvent correspondre à :
- Des échantillons gratuits envoyés aux clients
- Des ajustements comptables internes
- Des erreurs de saisie dans le CRM

Décision : Si les product_code associés sont des codes non-produits (frais de port, ajustements…), il s'agit probablement d'un cas métier légitime. Sinon, c'est une anomalie. Dans tous les cas, on les exclut du calcul du CA mais on les conserve dans le dataset.

In [27]:
zero_price = transactions[transactions["unit_price"] == 0]
print(f"Lignes unit_price == 0 : {len(zero_price)} ({len(zero_price)/len(transactions)*100:.1f}%)")
print("\n--- Échantillon unit_price == 0 ---")
print(zero_price[["invoice_id", "product_code", "product_name", "quantity", "unit_price"]].head(10))

Lignes unit_price == 0 : 10674 (0.6%)

--- Échantillon unit_price == 0 ---
     invoice_id product_code                         product_name  quantity  \
26       768176        22990                  WHITE CHERRY LIGHTS      24.0   
113      548380        22127                   Dotcom sold in 6's     -60.0   
136      734573        21145  CHARLIE + LOLA RED HOT WATER BOTTLE       1.0   
252      519368        37449                                  NaN       7.0   
261      508700        22192                                  NaN      -1.0   
585      656137        22156      3 GARDENIA MORRIS BOXED CANDLES       4.0   
730      628178        21507     SINGLE HEART ZINC T-LIGHT HOLDER      12.0   
842      758411        20659     GREETING CARD, OVERCROWDED POOL.      12.0   
1163     496720        85057                                  NaN      -1.0   
1185     561490        21275                                  NaN       3.0   

      unit_price  
26           0.0  
113          0.0 

 4. Codes produits atypiques (non-produits)

Les vrais produits ont généralement un code numérique (ex : 22382, 47591B). Certains codes correspondent à des frais de port (POST), ajustements (DOT, M, D), frais bancaires (BANK CHARGES), dons (CRUK), etc.

Décision : On les identifie avec une regex et on crée un flag is_non_product. Ces lignes sont exclues de l'analyse produit et panier mais restent dans le dataset.

In [28]:
print("--- Product codes les plus courts ou non-numériques ---")
transactions["pc_is_numeric"] = transactions["product_code"].astype(str).str.match(r"^\d+[A-Za-z]?$")
non_product = transactions[~transactions["pc_is_numeric"]]
print(non_product["product_code"].value_counts().head(20))
print(f"\nTotal lignes non-produit : {len(non_product)} ({len(non_product)/len(transactions)*100:.1f}%)")

--- Product codes les plus courts ou non-numériques ---
product_code
POST            2286
DOT             1647
M               1619
15056BL         1111
C2               489
79323LP          392
D                374
79323GR          324
BANK CHARGES     302
15056bl          288
S                281
ADJUST           256
DCGS0058         238
AMAZONFEE        236
gift_0001_20     227
CRUK             214
DCGS0066P        214
DCGSSGIRL        211
DCGS0076         207
DCGS0003         207
Name: count, dtype: int64

Total lignes non-produit : 20330 (1.1%)


In [29]:
transactions["is_non_product"] = ~transactions["pc_is_numeric"]

 5. Calcul de line_total et vérification de cohérence

On calcule line_total = quantity × unit_price pour chaque ligne, puis on vérifie la cohérence :
- Y a-t-il des line_total négatifs qui ne sont pas des retours ?
- Y a-t-il des line_total à zéro (lié aux unit_price == 0) ?

In [30]:
transactions["line_total"] = transactions["quantity"] * transactions["unit_price"]

print("--- Statistiques line_total ---")
print(transactions["line_total"].describe())

# Lignes incohérentes : line_total négatif sans être un retour flaggé
incoherent = transactions[(transactions["line_total"] < 0) & (~transactions["is_return"])]
print(f"\nLignes line_total < 0 mais pas retour : {len(incoherent)}")

# line_total == 0
print(f"Lignes line_total == 0 : {(transactions['line_total'] == 0).sum()}")

--- Statistiques line_total ---
count    1.820950e+06
mean     2.024693e+01
std      2.303198e+02
min     -1.684696e+05
25%      3.750000e+00
50%      1.008000e+01
75%      1.770000e+01
max      1.684696e+05
Name: line_total, dtype: float64

Lignes line_total < 0 mais pas retour : 5
Lignes line_total == 0 : 10590


---

## Customers.csv

In [31]:
print("=== CUSTOMERS : shape ===")
print(customers.shape)
print(customers.dtypes)
print(customers.describe())

=== CUSTOMERS : shape ===
(50000, 9)
customer_id         int64
country               str
first_purchase        str
last_purchase         str
n_orders          float64
total_spent       float64
avg_basket        float64
recency_days      float64
tenure_days       float64
dtype: object
        customer_id      n_orders   total_spent    avg_basket  recency_days  \
count  50000.000000  50000.000000  50000.000000  50000.000000  50000.000000   
mean   38390.294040      4.746364    421.538510     65.428104    238.454858   
std    14517.833799      9.551931   2014.135812    139.874935    218.933454   
min    12346.000000      0.850000      0.160000      0.160000      0.000000   
25%    25941.750000      1.060000     36.987500     21.787500     38.137500   
50%    38441.500000      2.220000     99.640000     38.810000    168.000000   
75%    50941.250000      5.210000    280.225000     69.100000    411.512500   
max    63441.000000    292.780000  69630.660000   5876.150000    844.590000   

   

 1. Cohérence des dates (first_purchase ≤ last_purchase)

On vérifie qu'aucun client n'a une date de premier achat postérieure à sa date de dernier achat. Si c'est le cas, il s'agit probablement d'une erreur d'export CRM.

Décision : On peut inverser les dates ou flagger ces lignes pour investigation.

In [32]:
customers["first_purchase"] = pd.to_datetime(customers["first_purchase"])
customers["last_purchase"] = pd.to_datetime(customers["last_purchase"])

date_incoherent = customers[customers["first_purchase"] > customers["last_purchase"]]
print(f"Clients avec first_purchase > last_purchase : {len(date_incoherent)}")
if len(date_incoherent) > 0:
    print(date_incoherent[["customer_id", "first_purchase", "last_purchase"]].head())

Clients avec first_purchase > last_purchase : 0


 2. Valeurs aberrantes (total_spent, n_orders, avg_basket)

On utilise la méthode IQR (Interquartile Range) pour détecter les outliers :
- IQR = Q3 − Q1
- Un point est considéré outlier s'il dépasse Q3 + 1.5 × IQR

Les outliers élevés en total_spent sont probablement des clients B2B ou grossistes.

Décision : On ne les supprime pas, mais on les flag (is_outlier_spent) car ils fausseraient les moyennes et les segmentations.

In [33]:
for col in ["total_spent", "n_orders", "avg_basket"]:
    Q1 = customers[col].quantile(0.25)
    Q3 = customers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = customers[(customers[col] < lower) | (customers[col] > upper)]
    print(f"{col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f} | Outliers: {len(outliers)} ({len(outliers)/len(customers)*100:.1f}%)")

print("\n--- Top 10 total_spent ---")
print(customers.nlargest(10, "total_spent")[["customer_id", "country", "n_orders", "total_spent", "avg_basket"]])

total_spent: Q1=36.99, Q3=280.23, IQR=243.24 | Outliers: 5827 (11.7%)
n_orders: Q1=1.06, Q3=5.21, IQR=4.15 | Outliers: 4059 (8.1%)
avg_basket: Q1=21.79, Q3=69.10, IQR=47.31 | Outliers: 3812 (7.6%)

--- Top 10 total_spent ---
       customer_id         country  n_orders  total_spent  avg_basket
928          21007  United Kingdom     84.36     69630.66  700.250000
32982        18714  United Kingdom     81.15     69290.75  572.770000
28931        19244  United Kingdom    108.00     67792.49  749.610000
43928        21275  United Kingdom     82.64     65002.13  608.670000
46727        18816  United Kingdom     88.34     62230.26  700.930000
40976        13694  United Kingdom     94.00     61703.62  656.421489
45480        21116  United Kingdom     96.28     61541.27  693.940000
47931        19541  United Kingdom     84.98     60581.52  653.470000
43804        19975  United Kingdom    106.68     60096.50  581.060000
48997        20209  United Kingdom     88.09     57066.60  694.410000


In [34]:
Q3_spent = customers["total_spent"].quantile(0.75)
IQR_spent = Q3_spent - customers["total_spent"].quantile(0.25)
customers["is_outlier_spent"] = customers["total_spent"] > (Q3_spent + 1.5 * IQR_spent)
print(f"Clients outliers total_spent : {customers['is_outlier_spent'].sum()}")

Clients outliers total_spent : 5827


 3. Clients one-shot vs récurrents

On distingue les clients ayant passé une seule commande (one-shot) des clients récurrents (plus d'une commande). Cette proportion est clé pour la stratégie marketing : un fort taux de one-shot peut indiquer un problème de rétention.

In [35]:
one_shot = customers[customers["n_orders"] <= 1]
recurrent = customers[customers["n_orders"] > 1]

print(f"Clients one-shot (1 commande) : {len(one_shot)} ({len(one_shot)/len(customers)*100:.1f}%)")
print(f"Clients récurrents (>1 commande) : {len(recurrent)} ({len(recurrent)/len(customers)*100:.1f}%)")

Clients one-shot (1 commande) : 9793 (19.6%)
Clients récurrents (>1 commande) : 40207 (80.4%)


---

## Résumé — Data Quality Report

Récapitulatif des anomalies détectées et des décisions prises :

In [36]:
print("=" * 60)
print("DATA QUALITY REPORT - ÉTAPE 2")
print("=" * 60)
print(f"Transactions : {len(transactions)} lignes")
print(f"  - customer_id manquants : {missing_cid} ({pct_missing:.1f}%)")
print(f"  - Retours (qty < 0) : {transactions['is_return'].sum()}")
print(f"  - unit_price == 0 : {(transactions['unit_price'] == 0).sum()}")
print(f"  - Non-produits : {transactions['is_non_product'].sum()}")
print(f"  - line_total calculé ✓")
print(f"\nCustomers : {len(customers)} lignes")
print(f"  - Dates incohérentes : {len(date_incoherent)}")
print(f"  - Outliers total_spent : {customers['is_outlier_spent'].sum()}")
print(f"  - One-shot : {len(one_shot)} | Récurrents : {len(recurrent)}")

DATA QUALITY REPORT - ÉTAPE 2
Transactions : 1837137 lignes
  - customer_id manquants : 418258 (22.8%)
  - Retours (qty < 0) : 23314


  - unit_price == 0 : 10674
  - Non-produits : 20330
  - line_total calculé ✓

Customers : 50000 lignes
  - Dates incohérentes : 0
  - Outliers total_spent : 5827
  - One-shot : 9793 | Récurrents : 40207
